### 랭체인 준비

In [1]:
# 패키지 설치
!pip install langchain
!pip install langchain-google-genai
!pip install langchain_community
!pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.7/408.7 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.13
    Uninstalling langchain-core-0.3.13:
      Successfully uninstalled langchain-core-0.3.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: SQLAlchemy
    Found existing installation: SQLAlchemy 2.0.36
    Uninstalling SQLAlchemy-2.0.36:
      Successfully uninstalled SQLAlchemy-2.0.36
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.4
    Uninstalling langchain-0.3.4:
      Successfully uninstalled langchain-0.3.4
   ━━━━━━━━

In [2]:
from google.colab import userdata
import os

# 환경 변수 준비(좌측 상단의 열쇠 아이콘으로 GOOGLE_API_KEY 설정)
os.environ["GOOGLE_API_KEY"]=userdata.get("GOOGLE_API_KEY")

### LLM

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 준비
llm = ChatGoogleGenerativeAI(
    model="models/gemini-1.5-flash",
)

In [4]:
# LLM 실행
messages = [
    ("human", "컴퓨터 게임을 만드는 새로운 회사 이름을 하나만 제안해주세요.")
]
output = llm.invoke(messages)
print(type(output))
print(output)

<class 'langchain_core.messages.ai.AIMessage'>
content='**게임 개발 회사 이름:**\n\n**픽셀리움(Pixellium)** \n\n이 이름은 "픽셀"과 "앨범"이라는 단어를 결합한 것으로, 게임 개발의 기본 요소인 픽셀과 게임 세계의 창의적 표현을 의미합니다. \n\n이 이름은 독특하고 기억하기 쉽고, 게임 개발에 대한 열정을 반영합니다.\n' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]} id='run-691b161d-f21e-4670-a298-a710ab597a9d-0' usage_metadata={'input_tokens': 22, 'output_tokens': 93, 'total_tokens': 115, 'input_token_details': {'cache_read': 0}}


### PromptTemplate

In [5]:
from langchain.prompts import ChatPromptTemplate

# PromptTemplate 준비
prompt_template = ChatPromptTemplate.from_messages([
    ("human", "{product}을 만드는 새로운 회사 이름을 하나만 제안해주세요."),
])

In [6]:
# PromptTemplate 실행
output = prompt_template.invoke({"product": "가정용 로봇"})
print(type(output))
print(output)

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[HumanMessage(content='가정용 로봇을 만드는 새로운 회사 이름을 하나만 제안해주세요.', additional_kwargs={}, response_metadata={})]


### OutputParser

In [7]:
from langchain_core.output_parsers import StrOutputParser

# OutputParser 준비
output_parser = StrOutputParser()

In [8]:
from langchain.schema import AIMessage

# OutputParser 실행
message = AIMessage(content="AI로부터의 메시지입니다.")
output = output_parser.invoke(message)
print(type(output))
print(output)

<class 'str'>
AI로부터의 메시지입니다.


### Chain

In [22]:
# Chain 준비
chain = prompt_template | llm | output_parser

In [23]:
# Chain 실행
output = chain.invoke({"product": "가정용 로봇"})
print(type(output))
print(output)

<class 'str'>
**로보홈(RoboHome)** 



### Agent

In [24]:
from langchain_core.tools import tool

@tool
def multiply(x: float, y: float) -> float:
    """'x'와 'y'를 곱함"""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """'x'를 'y'로 거듭제곱"""
    return x**y

@tool
def add(x: float, y: float) -> float:
    """'x'와 'y'를 더함"""
    return x + y

In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 준비
llm = ChatGoogleGenerativeAI(
    model="models/gemini-1.5-flash",
)

In [26]:
# httpx 패키지 설치
!pip install httpx

In [27]:
from langgraph.prebuilt import chat_agent_executor

# Agent 준비
agent_executor = chat_agent_executor.create_tool_calling_executor(
    llm,
    tools=[multiply, exponentiate, add]
)

In [28]:
# Tool을 사용하지 않는 질의 응답
response = agent_executor.invoke(
    {"messages": [("human", "안녕하세요!")]}
)
response["messages"][-1].content

'안녕하세요! 무엇을 도와드릴까요? 😊 \n'

In [29]:
# Tool을 사용한 질의 응답
response = agent_executor.invoke(
    {"messages": [("human", "1에 2를 더하면?")]}
)
response["messages"][-1].content

'3입니다.'

In [30]:
# 중간 처리 확인
for message in response["messages"]:
    print(message)

content='1에 2를 더하면?' additional_kwargs={} response_metadata={} id='7187de73-48aa-4bb1-b714-d307cc1a92ee'
content='' additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"x": 1.0, "y": 2.0}'}} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}]} id='run-6a5e5154-396a-44fb-95bd-23951c09a746-0' tool_calls=[{'name': 'add', 'args': {'x': 1.0, 'y': 2.0}, 'id': '12fc339f-f320-4faa-9f89-30cbe2770876', 'type': 'tool_call'}] usage_metadata={'input_tokens': 167, 'output_tokens': 18, 'total_tokens': 185, 'input_token_details': {'cache_read': 0}}
co

In [31]:
# Tool을 두 번 사용한 질의 응답
response = agent_executor.invoke(
    {"messages": [("human", "(1+2)*3은?")]}
)
response["messages"][-1].content

'(1+2)*3은 9입니다.'

In [32]:
# 중간 처리 확인
for message in response["messages"]:
    print(message)

content='(1+2)*3은?' additional_kwargs={} response_metadata={} id='9f86128c-ce4b-4e21-90a1-15a4e5647f44'
content='' additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"x": 1.0, "y": 2.0}'}} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]} id='run-abf6ef16-a9ea-4416-8b61-4eb9ac44864e-0' tool_calls=[{'name': 'add', 'args': {'x': 1.0, 'y': 2.0}, 'id': 'dffa017a-5772-4172-91b1-eb51ce2ff10b', 'type': 'tool_call'}] usage_metadata={'input_tokens': 167, 'output_tokens': 18, 'total_tokens': 185, 'input_token_details': {'cache_read': 0}}
con

### LangSmith

In [33]:
%pip install -U langsmith > /dev/null

In [34]:
import os
from uuid import uuid4

# 환경 변수 준비
unique_id = uuid4().hex[0:8]
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"Tracing Walkthrough2 - {unique_id}"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")